## setup

In [ ]:
import numpy as np

from qick import *

from qick.asm_riscv import RvAveragerProgram as AveragerProgramV2
from qick.asm_riscv import RvProgram as QickProgramV2
from qick.asm_v2 import QickSweep1D
from qick.asm_v2_riscv import rv_reg, WMEM_FIELDS, WREG_RV

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, force=True, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')

In [ ]:
from qick.riscv_toolchain import CC
assert CC is not None, "riscv64-unknown-elf-clang (or -gcc) not found on PATH"

from qick.riscq_config import RvConfig
from qick.riscv import _socmap, DEFAULT_SOC
soc = soccfg = RvConfig(_socmap(DEFAULT_SOC))
print(soccfg)

In [ ]:
GEN_CH = 1
RO_CH = 0
TRIG_TIME = 0.30
FREQ = 599

# times are in ticks of the time counter: 2047 ticks is the biggest time that fits in an addi
TICKS_PER_US = soccfg['tprocs'][0]['f_time']
CYCLES_PER_US = soccfg['gens'][GEN_CH]['f_fabric']
SCRATCH = 't2'

def asm_lines(prog):
    return [l.strip() for l in prog.asm().splitlines() if not l.startswith('.equ')]

def has_seq(lines, *seq):
    n = len(seq)
    return any(tuple(lines[i:i+n]) == seq for i in range(len(lines) - n + 1))

def expect_error(match, f, *args, **kwargs):
    try:
        f(*args, **kwargs)
    except RuntimeError as e:
        assert match in str(e), f"wrong error: {e}"
        print("refused:", str(e).splitlines()[0])
    else:
        raise AssertionError(f"expected a RuntimeError containing {match!r}, but nothing was raised")

## 12-bit immediates

RV32I immediates are signed 12-bit, so `addi` takes [-2048, 2047].
Anything outside that range needs a `li` into the scratch register followed by a register `add`.

### register increments
Same idea as the register operations demo in 00_intro: a single data register, incremented by values on either side of both boundaries.

In [ ]:
prog = QickProgramV2(soccfg)
prog.add_reg(name='myreg')
prog.write_reg(dst='myreg', src=0)
for val in [2047, 2048, -2048, -2049]:
    prog.inc_reg(dst='myreg', src=val)
prog.end()

print("text ASM:")
print("\n".join(asm_lines(prog)))

In [ ]:
lines = asm_lines(prog)
r = rv_reg(prog, 'myreg')

# in range: one addi
assert f'addi {r}, {r}, 2047' in lines
assert f'addi {r}, {r}, -2048' in lines
# out of range: li into the scratch register, then add
assert has_seq(lines, f'li {SCRATCH}, 2048', f'add {r}, {r}, {SCRATCH}')
assert has_seq(lines, f'li {SCRATCH}, -2049', f'add {r}, {r}, {SCRATCH}')

### pulse times and delays
Pulse/trigger times are added to the reference time (`s11`) to get an absolute time in `t0`, and `delay()` adds to the reference time itself.
Both go through the same immediate check, so 2047 ticks (4.094 us) should be an `addi` and 2048 ticks (4.096 us) should not.

In [ ]:
class ImmTimeProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch,
                       style="const",
                       freq=cfg['freq'],
                       length=0.05,
                       phase=0,
                       gain=1.0,
                      )

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], t=cfg['trig_time'])
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=cfg['t'], tag='pulse')
        self.delay(cfg['t'], tag='delay')
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)

config = {'gen_ch': GEN_CH,
          'ro_ch': RO_CH,
          'freq': FREQ,
          'trig_time': TRIG_TIME,
          'ro_len': 0.3,
         }

for ticks in [2047, 2048]:
    config['t'] = ticks/TICKS_PER_US
    prog = ImmTimeProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
    lines = asm_lines(prog)

    assert prog.get_time_param('delay', 't') == config['t']

    if ticks < 2048:
        assert f'addi t0, s11, {ticks}' in lines           # pulse time
        assert f'addi s11, s11, {ticks}' in lines          # delay
    else:
        assert has_seq(lines, f'li {SCRATCH}, {ticks}', f'add t0, s11, {SCRATCH}')
        assert has_seq(lines, f'li {SCRATCH}, {ticks}', f'add s11, s11, {SCRATCH}')
    print("%d ticks (%.3f us): ok" % (ticks, config['t']))

### sweep steps
A sweep adds its step to a waveform register at the end of every loop pass, then subtracts `step + span` when the loop exits to put the register back where it started.
So the reset is the bigger of the two numbers, and the two can land on different sides of the boundary.
These are gain sweeps with 2 steps, so the raw step equals the raw span (gain is in units of `maxv`, 32766 for full scale).

In [ ]:
class SweepStepProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']

        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])

        self.add_loop("myloop", self.cfg["steps"])

        self.add_pulse(ch=gen_ch, name="mypulse", ro_ch=ro_ch,
                       style="const",
                       freq=cfg['freq'],
                       length=0.05,
                       phase=0,
                       gain=cfg['gain'],
                      )

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], t=cfg['trig_time'])
        self.pulse(ch=cfg['gen_ch'], name="mypulse", t=0)

MAXV = soccfg['gens'][GEN_CH]['maxv']
GAIN_REG = dict(zip(WMEM_FIELDS, WREG_RV))['gain']

config = {'steps': 2,
          'gen_ch': GEN_CH,
          'ro_ch': RO_CH,
          'freq': FREQ,
          'trig_time': TRIG_TIME,
          'ro_len': 0.3,
         }

def add_insts(val):
    if -2048 <= val <= 2047:
        return (f'addi {GAIN_REG}, {GAIN_REG}, {val}',)
    return (f'li {SCRATCH}, {val}', f'add {GAIN_REG}, {GAIN_REG}, {SCRATCH}')

for step in [1024, 2047, 2048]:
    config['gain'] = QickSweep1D("myloop", 0.0, step/MAXV)
    prog = SweepStepProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
    lines = asm_lines(prog)

    gains = prog.get_pulse_param('mypulse', 'gain', as_array=True)
    assert np.allclose(gains, [0, step/MAXV]), gains

    reset = -2*step
    assert has_seq(lines, *add_insts(step)), f"step {step} not found"
    assert has_seq(lines, *add_insts(reset)), f"reset {reset} not found"
    print("step %d, reset %d: ok" % (step, reset))

### shift amounts
`cond_jump(op='>>', arg2=n)` turns into `srai t0, reg, n`, and the `srai` shift field is only 5 bits wide, so n has to be in [0, 31].
The translator only checks that n fits in 12 bits, so a shift of 32 to 2047 gets as far as the assembler, which should refuse it.

In [ ]:
def shift_program(n):
    prog = QickProgramV2(soccfg)
    prog.add_reg(name='myreg')
    prog.write_reg(dst='myreg', src=-1)
    prog.label('top')
    prog.cond_jump('top', 'myreg', 'Z', op='>>', arg2=n)
    prog.end()
    return prog

prog = shift_program(31)
r = rv_reg(prog, 'myreg')
assert f'srai t0, {r}, 31' in asm_lines(prog)

expect_error("failed", shift_program(32).compile)